# KV Cache Compression Benchmark Pipeline

This notebook allows you to automatically execute and compare multiple KV cache compression techniques across LongBench and RULER benchmarks.

### "Ours" vs "CommonKV"
In this repository, **"ours"** and **"commonkv"** refer to the exact same methodology (the paper's proposed cross-layer SVD sharing). The codebase uses the flag `commonkv` when patching LLama models and `ours` when patching Mistral models for historical reasons.



In [1]:
!git clone https://github.com/Seqaeon/CommonKV.git


Cloning into 'CommonKV'...
remote: Enumerating objects: 631, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 631 (delta 0), reused 0 (delta 0), pack-reused 626 (from 2)
Receiving objects: 100% (631/631), 621.64 KiB | 5.45 MiB/s, done.
Resolving deltas: 100% (409/409), done.


In [2]:
!git pull origin main


fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [3]:
import os
os.chdir("/kaggle/working/CommonKV")
print(os.getcwd())

/kaggle/working/CommonKV


In [4]:
import os
import sys
import subprocess
import time
import json
import torch
import pandas as pd
from IPython.display import display, HTML

# Add repo path
sys.path.append(os.path.abspath('.'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')



Using device: cuda


In [5]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
import os
os.environ["HF_TOKEN"] = secret_value_0 #"your_hf_token_here"


## 1. Define Pipeline Configuration


In [6]:
# MODEL_PATH =  "/kaggle/input/models/metaresearch/llama-3.1/transformers/8b-instruct/2" #'meta-llama/Meta-Llama-3.1-8B-Instruct'

# MODEL_PATH = "Qwen/Qwen2.5-0.5B-Instruct" # 'meta-llama/Meta-Llama-3.1-8B-Instruct'
# MODEL_PATH = "mistralai/Mistral-7B-Instruct-v0.2"
# MODEL_PATH = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MODEL_PATH = "meta-llama/Llama-3.2-3B-Instruct"
# DATASET = 'narrativeqa' # Example LongBench dataset. Options: qasper, multifieldqa_en, hotpotqa, etc.
DATASET = 'all' # Set to 'all' to evaluate multiple datasets via --max_datasets

# The list of all compression methods to evaluate
# Options: 'fullkv', 'commonkv' (llama), 'ours' (mistral), 'think', 'palu', 'minicache', 'snapkv', 'pyramidkv', 'h2o', 'my_custom_method'
METHODS_TO_TEST = ["apkvc", 'commonkv' ,'fullkv', 'think', 'palu', 'minicache']

# # CommonKV / Baseline Parameters
# RANK = 4096
# LAYER_STEP = 2
# MAX_CAPACITY_PROMPTS = 8950



In [7]:
# --- Recommended Research Configuration ---
# MODEL_PATH = "alpindale/Mistral-7B-v0.2-hf" 
MAX_CAPACITY_PROMPTS = 1024  # Standard budget for most papers
RANK = 64                    # 2x compression for Palu/SVD
LAYER_STEP = 1               # Apply to every layer for consistency

# APKVC Specifics (to match the 1024 budget above)
MAX_ANCHOR_INTERVAL = 16 
PREDICTOR = "linear"         # Use 'linear' for much higher accuracy



## 2. Execute Benchmark Pipeline
Because the repository relies on **monkeypatching** the HuggingFace `transformers` global namespace to inject the different KV caching techniques, loading multiple methods sequentially in the same Python process will cause collisions and crashes.

To safely and programatically run all of them, we execute the repo's evaluation scripts (`run_longbench.py` and `run_ruler.py`) in isolated subprocesses.

**Note**: To evaluate `my_custom_method` on these benchmarks, you will need to add your custom logic hooks inside `commonkv/monkeypatch.py` so the `run_longbench.py` script can pick it up!



In [8]:
!sed -i 's/from modeling_llama/from commonkv.modeling_llama/g' commonkv/monkeypatch.py
!sed -i 's/from modeling_mistral/from commonkv.modeling_mistral/g' commonkv/monkeypatch.py
!sed -i 's/from llama_model/from commonkv.llama_model/g' commonkv/monkeypatch.py
!sed -i 's/from mistral_model/from commonkv.mistral_model/g' commonkv/monkeypatch.py


In [9]:
# %%bash
# git clone https://github.com/Zefan-Cai/PyramidKV.git /tmp/PyramidKV
# # If the repo contains an inner pyramidkv folder, extract it. 
# # Otherwise, treat the entire repo as the pyramidkv module itself!
# if [ -d "/tmp/PyramidKV/pyramidkv" ]; then 
#     cp -r /tmp/PyramidKV/pyramidkv ./
# else 
#     cp -r /tmp/PyramidKV ./pyramidkv
# fi


In [10]:
%%bash
# 1. Force the exact transformers version (>=4.44) the authors wrote patches for 
#    before Hugging Face removed `SlidingWindowCache` in 4.45+
pip install "transformers==4.44.2"

# 2. Install flash-attention (Required by llama_model.py)
#    Note: Compiling this from source might take ~10-15 minutes on Kaggle!
# pip install flash-attn --no-build-isolation

# 3. Clone PyramidKV
git clone https://github.com/Zefan-Cai/PyramidKV.git /tmp/PyramidKV

# 4. Generate a dynamic setup.py for it! 
#    Because the authors didn't include one, we'll force Python to recognize 
#    the root scripts or inner folders natively as a named module.
cat << 'EOF' > /tmp/PyramidKV/setup.py
from setuptools import setup, find_packages
import os

# Grab all standalone python scripts in the repo to bundle as modules
modules = [f[:-3] for f in os.listdir('.') if f.endswith('.py') and f != 'setup.py']

setup(
    name="pyramidkv",
    version="0.1.0",
    packages=find_packages(),
    py_modules=modules
)
EOF

# 5. Install it as a native python package using our minted setup.py
pip install -e /tmp/PyramidKV


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Obtaining file:///tmp/PyramidKV
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.

Cloning into '/tmp/PyramidKV'...
Updating files: 100% (166/166), done.


In [11]:
from datasets import load_dataset, Dataset
context_length = 4096
ds = load_dataset("simonjegou/ruler", str(context_length) , split="test")
# print(ds[0])
# ds.to_json("ruler_16384.jsonl")
import os

os.makedirs(f"data/RULER/{context_length}", exist_ok=True)

for task_name in set(ds["task"]):
    subset = ds.filter(lambda ex: ex["task"] == task_name)
    out_path = f"data/RULER/{context_length}/{task_name}.jsonl"
    subset.to_json(out_path)
    print(f"Saved {task_name} → {out_path}")





README.md: 0.00B [00:00, ?B/s]

4096/test-00000-of-00001.parquet:   0%|          | 0.00/24.4M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/6500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6500 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved qa_1 → data/RULER/4096/qa_1.jsonl


Filter:   0%|          | 0/6500 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved fwe → data/RULER/4096/fwe.jsonl


In [12]:
from huggingface_hub import hf_hub_download
import zipfile


file_path = hf_hub_download(
    repo_id="zai-org/LongBench",
    filename="data.zip",
    repo_type="dataset"
)


# zip_path = file_path   # path returned by hf_hub_download
# extract_dir = "./data/LongBench"

import os, zipfile

zip_path = file_path
extract_dir = "./data/LongBench"   # target root directory

with zipfile.ZipFile(zip_path, "r") as z:
    for member in z.namelist():
        # Strip the leading "data/" from paths inside the zip
        if member.startswith("data/"):
            relative_path = member[len("data/"):]  # remove "data/"
        else:
            relative_path = member

        # Skip empty directory entries
        if not relative_path:
            continue

        target_path = os.path.join(extract_dir, relative_path)

        # Make sure subdirectories exist
        os.makedirs(os.path.dirname(target_path), exist_ok=True)

        # Extract file contents
        if not member.endswith("/"):  # skip directories
            with z.open(member) as source, open(target_path, "wb") as target:
                target.write(source.read())



data.zip:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [13]:
APKVC_CALIB_PATH = "./artifacts/apkvc/tinyllama_codebooks.pt"
USE_APKVC_CALIB = True
apkvc_use_rope_aware_aq = 1
import os, subprocess
os.makedirs("./traces", exist_ok=True)
APKVC_TRACE_PATH = "./traces/lb_mix.pt"
# LongBench trace dump
cmd_lb = [
    "python3", "run_longbench.py",
    "--method", "apkvc",
    "--model_path", MODEL_PATH,
    "--attn_implementation", "sdpa",
    "--dataset", "hotpotqa",
    "--steps", "100",
    "--save_dir", "./tmp_trace_run_lb",
    "--apkvc_use_rope_aware_aq", str(apkvc_use_rope_aware_aq),   # disable
    "--apkvc_trace_output_path", str(APKVC_TRACE_PATH),
    "--apkvc_trace_max_samples", "400000",
    "--apkvc_trace_chunk_size", "0",
    "--enforce_uniform_prefill_cap", "1",
    "--eval_batch_size", "1",
    "--max_prefill_tokens_for_custom_methods", "7950"
]
subprocess.run(cmd_lb, check=True)
print("LongBench trace created: ./traces/lb_mix.pt")



The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.18s/it]


Loading data...
Max Length is 12697
Finish loading model and tokenizer


  0%|          | 0/200 [00:00<?, ?it/s]2026-03-28 06:41:51.720785: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774680112.141802     369 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774680112.245185     369 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774680113.190468     369 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774680113.190508     369 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774680113.190511     369 computatio

[APKVC] dumped trace residuals -> ./traces/lb_mix.pt
LongBench trace created: ./traces/lb_mix.pt


In [14]:
import os, subprocess
os.makedirs("./artifacts/apkvc", exist_ok=True)

trace_paths = [
    "./traces/lb_mix.pt",
    # "./traces/ruler_16k.pt",
]
for p in trace_paths:
    print(p, "exists?", os.path.exists(p))
calib_cmd = [
    "python3", "scripts/calibrate_apkvc_codebooks.py",
    "--trace_paths", "./traces/lb_mix.pt",
    "--output_path", APKVC_CALIB_PATH,
    "--K_num_codebooks", "4",
    "--V_num_codebooks", "2",
    "--codebook_size", "256",
    "--max_samples", "400000",
    "--chunk_size", "2048",
    "--device", "cuda",
]
print("Running:", " ".join(calib_cmd))
subprocess.run(calib_cmd, check=True)
print("Saved:", APKVC_CALIB_PATH)

./traces/lb_mix.pt exists? True
Running: python3 scripts/calibrate_apkvc_codebooks.py --trace_paths ./traces/lb_mix.pt --output_path ./artifacts/apkvc/tinyllama_codebooks.pt --K_num_codebooks 4 --V_num_codebooks 2 --codebook_size 256 --max_samples 400000 --chunk_size 2048 --device cuda
Saved calibrated codebooks -> ./artifacts/apkvc/tinyllama_codebooks.pt
Saved: ./artifacts/apkvc/tinyllama_codebooks.pt


In [15]:
# %%bash
# python3 scripts/calibrate_apkvc_codebooks.py \
#   --trace_paths ./traces/lb_mix.pt \
#   --output_path ./artifacts/apkvc/tinyllama_codebooks.pt \
#   --K_num_codebooks 4 \
#   --V_num_codebooks 2 \
#   --codebook_size 64 \
#   --max_samples 400000 \
#   --chunk_size 2048

In [16]:
# import torch
# p = "./artifacts/apkvc/tinyllama_codebooks.pt"
# obj = torch.load(p, map_location="cpu")
# print(type(obj))
# print(obj.keys() if isinstance(obj, dict) else "not dict")

In [17]:
longbench_results_dir = "./results_long_bench"
METHODS_TO_TEST = ["apkvc", "fullkv"]

os.makedirs(longbench_results_dir, exist_ok=True)

DUMP_APKVC_TRACES = False
USE_APKVC_CALIB = True
APKVC_CALIB_PATH = "./artifacts/apkvc/tinyllama_codebooks.pt"
APKVC_TRACE_PATH = "./traces/lb_mix.pt"

for method in METHODS_TO_TEST:
    print(f"\n{'='*50}\nEvaluating Method: {method.upper()}\n{'='*50}")
    eval_method = method

    # cmd = [
    #     "python3", "run_longbench.py",
    #     "--method", eval_method,
    #     "--model_path", MODEL_PATH,
    #     "--dataset", DATASET,
    #     "--max_capacity_prompts", str(MAX_CAPACITY_PROMPTS),
    #     "--attn_implementation", "sdpa",
    #     "--rank", str(RANK),
    #     "--layer_step", str(LAYER_STEP),
    #     "--save_dir", longbench_results_dir,
    #     "--eval_batch_size", "1",
    #     "--max_datasets", "2",
    #     "--steps", "20",
    #     "--enforce_uniform_prefill_cap", "1",
    #     "--max_prefill_tokens_for_custom_methods", "32768",
    #     "--truncation_policy", "head_tail",
    # ]
    cmd = [
        "python3", "run_longbench.py",
        "--method", eval_method,
        "--model_path", MODEL_PATH,
        "--dataset", DATASET,
        "--max_capacity_prompts", str(MAX_CAPACITY_PROMPTS),
        "--attn_implementation", "sdpa",
        "--rank", str(RANK),
        "--layer_step", str(LAYER_STEP),
        "--max_anchor_interval", str(MAX_ANCHOR_INTERVAL),
        "--predictor_type", PREDICTOR,
        "--save_dir", longbench_results_dir,
        "--apkvc_use_rope_aware_aq", str(apkvc_use_rope_aware_aq),   # disable
        "--eval_batch_size", "1",
        "--max_datasets", "2",
        "--steps", "20",
        "--enforce_uniform_prefill_cap", "1",
        "--max_prefill_tokens_for_custom_methods", "7950",# "32768",
        "--truncation_policy", "head_tail",

    ]

    if eval_method.lower() in ["apkvc", "custom"]:
        cmd.extend([
            "--max_anchor_interval", str(MAX_ANCHOR_INTERVAL),
            "--predictor_type", PREDICTOR,
            "--apkvc_use_rope_aware_aq", str(apkvc_use_rope_aware_aq),
        ])

        if DUMP_APKVC_TRACES:
            cmd.extend([
                "--apkvc_trace_output_path", APKVC_TRACE_PATH,
                "--apkvc_trace_max_samples", "400000",
            ])

        if USE_APKVC_CALIB:
            cmd.extend(["--apkvc_calibration_path", APKVC_CALIB_PATH])

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    process.wait()
    print(f"\nReturn code: {process.returncode}")


Evaluating Method: APKVC

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.19s/it]
Working on max_capacity_prompts 1024 dataset narrativeqa - 1/2
Loading data...
Max Length is 36418
Finish loading model and tokenizer

  0%|          | 0/200 [00:00<?, ?it/s]2026-03-28 07:13:13.382570: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774681993.401343     927 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774681993.407155     927 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774681993.424460     927 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more

## 3. Compare Results
The `run_longbench.py` script dumps `.json` files containing the model's generated text predictions. 
The official LongBench metrics (like F1 score, Exact Match, etc.) require the `eval.py` script from the official THUDM/LongBench repository. 

However, we can look at the generations and lengths directly generated from our pipeline loop!



In [18]:
!pip install rouge
!pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 3.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.3 MB/s eta 0:00:0000:0100:01


In [19]:
import os, subprocess, pandas as pd

# results_dir = "/kaggle/working/CommonKV/results_long_bench/Mistral-7B-Instruct-v0.2_64_1_v4"
results_dir = "/kaggle/working/CommonKV/results_long_bench/Llama-3.2-3B-Instruct_64_1_v4"
# results_dir = "/kaggle/working/CommonKV/results_long_bench/TinyLlama-1.1B-Chat-v1.0_64_1_v4"

eval_py = "/kaggle/working/CommonKV/eval.py"

# 1. Run the evaluation once for all discovered methods
subprocess.run(["python3", eval_py, "--results_dir", results_dir])

# 2. Load the unified results.csv that eval.py now generates automatically
csv_path = os.path.join(results_dir, "results.csv")
if os.path.exists(csv_path):
    # Melt the broad table into the [dataset, method, score] format you wanted
    df = pd.read_csv(csv_path)
    scores_df = df.melt(id_vars=["dataset"], var_name="method", value_name="score")
    
    # Filter out empty results (-1) or methods that weren't evaluates
    scores_df = scores_df[scores_df["score"] != -1]
    
    display(scores_df.sort_values(["dataset", "method"]))
else:
    print("Evaluation failed to generate results.csv")


/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


Datasets to evaluate: ['narrativeqa', 'qasper']
Methods to evaluate: ['FullKV', 'apkvc']
dataset narrativeqa method FullKV scores {'narrativeqa': 20.56, 'compression_ratio': np.float64(1.0), 'latency': np.float64(15.567804999999998), 'tps': np.float64(0.52356)}
dataset narrativeqa method apkvc scores {'narrativeqa': 3.48, 'compression_ratio': np.float64(0.06800000000000002), 'latency': np.float64(22.942605000000004), 'tps': np.float64(4.429150000000001)}
dataset qasper method FullKV scores {'qasper': 34.48, 'compression_ratio': np.float64(1.0), 'latency': np.float64(16.98771), 'tps': np.float64(1.8191)}
dataset qasper method apkvc scores {'qasper': 5.08, 'compression_ratio': np.float64(0.06800000000000002), 'latency': np.float64(23.4538), 'tps': np.float64(5.43309)}

      TIE TABLE (Win / Loss / Tie)
                     FullKV       apkvc
FullKV                    -       0/0/0
apkvc                 0/0/0           -



100%|██████████| 2/2 [00:00<00:00, 38.78it/s]


,dataset,method,score
0,narrativeqa,FullKV,20.56
2,narrativeqa,apkvc,3.48
1,qasper,FullKV,34.48
3,qasper,apkvc,5.08


In [20]:
import os, pandas as pd

# results_dir = "/kaggle/working/CommonKV/results_long_bench/TinyLlama-1.1B-Chat-v1.0_4096_2_v4"
jsonl_path = os.path.join(results_dir, "results_long.jsonl")

if os.path.exists(jsonl_path):
    # 1. Read the machine-readable long format directly
    df = pd.read_json(jsonl_path, lines=True)
    
    # 2. Filter out missing results (-1)
    df = df[df["score"] != -1]
    
    # 3. Sort and Display (Score and CR are now in separate columns)
    display(df.sort_values(["dataset", "method"]))
else:
    print("Evaluation failed to generate results_long.jsonl")


,dataset,method,score,compression_ratio,latency,tps
0,narrativeqa,FullKV,20.56,1.000,15.5678,0.52
1,narrativeqa,apkvc,3.48,0.068,22.9426,4.43
2,qasper,FullKV,34.48,1.000,16.9877,1.82
3,qasper,apkvc,5.08,0.068,23.4538,5.43


In [21]:
# RULER trace dump
APKVC_CALIB_PATH = "./artifacts/apkvc/tinyllama_codebooks02.pt"
USE_APKVC_CALIB = True

cmd_ruler = [
    "python3", "run_ruler.py",
    "--method", "apkvc",
    "--model_path", MODEL_PATH,
    "--dataset", "niah_single_1",
    "--context_lengths", "4096",
    "--attn_implementation", "sdpa",
    "--steps", "100",
    "--save_dir", "./tmp_trace_run_ruler",
    "--apkvc_trace_output_path", "./traces/ruler_16k.pt",
    "--apkvc_trace_max_samples", "400000",
    "--enforce_uniform_prefill_cap", "1",
    "--eval_batch_size", "1",
    "--max_prefill_tokens_for_custom_methods", "7950"

]
subprocess.run(cmd_ruler, check=True)
print("RULER trace created: ./traces/ruler_16k.pt")

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.10s/it]


Working on max_capacity_prompts 512 dataset niah_single_1 - Context 4096
Loading data...
Max Length is 13802
Finish loading model and tokenizer


  0%|          | 0/500 [00:00<?, ?it/s]2026-03-28 07:40:53.359997: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774683653.385336    1002 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774683653.393323    1002 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774683653.414194    1002 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774683653.414221    1002 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774683653.414224    1002 computatio

[APKVC] dumped trace residuals -> ./traces/ruler_16k.pt
RULER trace created: ./traces/ruler_16k.pt


In [22]:
import os, subprocess
os.makedirs("./artifacts/apkvc", exist_ok=True)

trace_paths = [
    "./traces/ruler_16k.pt",
    # "./traces/ruler_16k.pt",
]
for p in trace_paths:
    print(p, "exists?", os.path.exists(p))
calib_cmd = [
    "python3", "scripts/calibrate_apkvc_codebooks.py",
    "--trace_paths", "./traces/ruler_16k.pt",
    "--output_path", APKVC_CALIB_PATH,
    "--K_num_codebooks", "4",
    "--V_num_codebooks", "2",
    "--codebook_size", "256",
    "--max_samples", "400000",
    "--chunk_size", "2048",
    "--device", "cuda",
]
print("Running:", " ".join(calib_cmd))
subprocess.run(calib_cmd, check=True)
print("Saved:", APKVC_CALIB_PATH)

./traces/ruler_16k.pt exists? True
Running: python3 scripts/calibrate_apkvc_codebooks.py --trace_paths ./traces/ruler_16k.pt --output_path ./artifacts/apkvc/tinyllama_codebooks02.pt --K_num_codebooks 4 --V_num_codebooks 2 --codebook_size 256 --max_samples 400000 --chunk_size 2048 --device cuda
Saved calibrated codebooks -> ./artifacts/apkvc/tinyllama_codebooks02.pt
Saved: ./artifacts/apkvc/tinyllama_codebooks02.pt


In [23]:
ruler_results_dir = "./results_ruler"
METHODS_TO_TEST = ["apkvc", "fullkv"]
MAX_CAPACITY_PROMPTS = 1024
os.makedirs(ruler_results_dir, exist_ok=True)

DUMP_APKVC_TRACES = False
USE_APKVC_CALIB = True
APKVC_CALIB_PATH = "./artifacts/apkvc/tinyllama_codebooks02.pt"
APKVC_TRACE_PATH = "./traces/ruler_16k.pt"

for method in METHODS_TO_TEST:
    print(f"\n{'='*50}\nEvaluating Method: {method.upper()}\n{'='*50}")
    eval_method = method

    # cmd = [
    #     "python3", "run_longbench.py",
    #     "--method", eval_method,
    #     "--model_path", MODEL_PATH,
    #     "--dataset", DATASET,
    #     "--max_capacity_prompts", str(MAX_CAPACITY_PROMPTS),
    #     "--attn_implementation", "sdpa",
    #     "--rank", str(RANK),
    #     "--layer_step", str(LAYER_STEP),
    #     "--save_dir", longbench_results_dir,
    #     "--eval_batch_size", "1",
    #     "--max_datasets", "2",
    #     "--steps", "20",
    #     "--enforce_uniform_prefill_cap", "1",
    #     "--max_prefill_tokens_for_custom_methods", "32768",
    #     "--truncation_policy", "head_tail",
    # ]
    cmd = [
        "python3", "run_ruler.py",
        "--method", eval_method,
        "--model_path", MODEL_PATH,
        "--dataset", DATASET,
        "--max_capacity_prompts", str(MAX_CAPACITY_PROMPTS),
        "--attn_implementation", "sdpa",
        "--rank", str(RANK),
        "--layer_step", str(LAYER_STEP),
        "--max_anchor_interval", str(MAX_ANCHOR_INTERVAL),
        "--predictor_type", PREDICTOR,
        "--save_dir", ruler_results_dir,
        "--apkvc_use_rope_aware_aq", str(apkvc_use_rope_aware_aq),   # disable
        "--eval_batch_size", "1",
        "--context_lengths", "4096",
        "--max_datasets", "2",
        "--steps", "20",
        "--enforce_uniform_prefill_cap", "1",
        "--max_prefill_tokens_for_custom_methods", "7950",# "32768",
        "--truncation_policy", "head_tail",

    ]

    if eval_method.lower() in ["apkvc", "custom"]:
        cmd.extend([
            "--max_anchor_interval", str(MAX_ANCHOR_INTERVAL),
            "--predictor_type", PREDICTOR,
            "--apkvc_use_rope_aware_aq", str(apkvc_use_rope_aware_aq),
        ])

        if DUMP_APKVC_TRACES:
            cmd.extend([
                "--apkvc_trace_output_path", APKVC_TRACE_PATH,
                "--apkvc_trace_max_samples", "400000",
            ])

        if USE_APKVC_CALIB:
            cmd.extend(["--apkvc_calibration_path", APKVC_CALIB_PATH])

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    process.wait()
    print(f"\nReturn code: {process.returncode}")


Evaluating Method: APKVC

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.12s/it]
Working on max_capacity_prompts 1024 dataset niah_single_1 - 1/2 - Context 4096
Loading data...
Max Length is 13802
Finish loading model and tokenizer

  0%|          | 0/500 [00:00<?, ?it/s]2026-03-28 08:13:08.323370: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774685588.348856    1039 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774685588.357685    1039 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774685588.379359    1039 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the

In [24]:
import os, subprocess, pandas as pd

# results_dir = "/kaggle/working/CommonKV/results_long_bench/Mistral-7B-Instruct-v0.2_64_1_v4"
results_dir = "/kaggle/working/CommonKV/results_ruler/Llama-3.2-3B-Instruct_1024/4096"
# results_dir = "/kaggle/working/CommonKV/results_long_bench/TinyLlama-1.1B-Chat-v1.0_64_1_v4"

eval_py = "/kaggle/working/CommonKV/eval_ruler.py"

# 1. Run the evaluation once for all discovered methods
subprocess.run(["python3", eval_py, "--results_dir", results_dir])

# 2. Load the unified results.csv that eval.py now generates automatically
csv_path = os.path.join(results_dir, "results.csv")
if os.path.exists(csv_path):
    # Melt the broad table into the [dataset, method, score] format you wanted
    df = pd.read_csv(csv_path)
    scores_df = df.melt(id_vars=["dataset"], var_name="method", value_name="score")
    
    # Filter out empty results (-1) or methods that weren't evaluates
    scores_df = scores_df[scores_df["score"] != -1]
    
    display(scores_df.sort_values(["dataset", "method"]))
else:
    print("Evaluation failed to generate results.csv")


Datasets to evaluate: ['niah_single_1', 'niah_single_2']
Methods to evaluate: ['FullKV', 'apkvc']

[DONE] Evaluation complete. Summary saved to: /kaggle/working/CommonKV/results_ruler/Llama-3.2-3B-Instruct_1024/4096
- Wide scores: results.csv
- Wide ratios: compression_ratios.csv
- Long format: results_ruler.jsonl


100%|██████████| 2/2 [00:00<00:00, 155.86it/s]


,dataset,method,score
0,niah_single_1,FullKV,100.0
2,niah_single_1,apkvc,95.0
1,niah_single_2,FullKV,100.0
3,niah_single_2,apkvc,85.0


In [25]:
!python3 eval_ruler.py --results_dir /kaggle/working/CommonKV/results_ruler/Llama-3.2-3B-Instruct_1024/4096

Datasets to evaluate: ['niah_single_1', 'niah_single_2']
Methods to evaluate: ['FullKV', 'apkvc']
100%|████████████████████████████████████████████| 2/2 [00:00<00:00, 160.55it/s]

[DONE] Evaluation complete. Summary saved to: /kaggle/working/CommonKV/results_ruler/Llama-3.2-3B-Instruct_1024/4096
- Wide scores: results.csv
- Wide ratios: compression_ratios.csv
- Long format: results_ruler.jsonl


In [26]:
import os, pandas as pd

# results_dir = "/kaggle/working/CommonKV/results_long_bench/TinyLlama-1.1B-Chat-v1.0_4096_2_v4"
jsonl_path = os.path.join(results_dir, "results_ruler.jsonl")

if os.path.exists(jsonl_path):
    # 1. Read the machine-readable long format directly
    df = pd.read_json(jsonl_path, lines=True)
    
    # 2. Filter out missing results (-1)
    df = df[df["score"] != -1]
    
    # 3. Sort and Display (Score and CR are now in separate columns)
    display(df.sort_values(["dataset", "method"]))
else:
    print("Evaluation failed to generate results_rule.jsonl")


,dataset,method,score,compression_ratio,latency,tps
0,niah_single_1,FullKV,100,1.000,14.0604,0.68
1,niah_single_1,apkvc,95,0.068,18.0532,3.55
2,niah_single_2,FullKV,100,1.000,17.5045,2.70
3,niah_single_2,apkvc,85,0.068,18.3025,3.50


In [27]:
# import json
# a = [json.loads(x) for x in open("/kaggle/working/CommonKV/results_long_bench/Mistral-7B-Instruct-v0.2_4096_2_v4/narrativeqa/minicache.json")]
# b = [json.loads(x) for x in open("/kaggle/working/CommonKV/results_long_bench/Mistral-7B-Instruct-v0.2_4096_2_v4/narrativeqa/palu.json")]
# print(sum(x["pred"]==y["pred"] for x,y in zip(a,b)), "identical preds out of", len(a))

In [28]:
# !sh scripts/scripts_longBench/metrics.sh /kaggle/working/CommonKV/results_long_bench/Mistral-7B-Instruct-v0.2_4096_2_v4 /kaggle/working/CommonKV/eval.py

In [29]:
from transformers import AutoConfig, AutoModelForCausalLM
from commonkv.monkeypatch import replace_llama, replace_mistral

def inspect_method(method, model_path):
    m = method.lower()
    replace_llama(m); replace_mistral(m)
    cfg = AutoConfig.from_pretrained(model_path, use_cache=True)
    model = AutoModelForCausalLM.from_pretrained(model_path, config=cfg, torch_dtype="auto", device_map="auto")
    attn = model.model.layers[0].self_attn
    print(method, "attn_class=", attn.__class__.__name__,
          "window_size=", getattr(attn.config, "window_size", None),
          "max_capacity_prompt=", getattr(attn.config, "max_capacity_prompt", None))
    del model

inspect_method("FullKV", MODEL_PATH)
inspect_method("ThinK", MODEL_PATH)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

FullKV attn_class= LlamaSdpaAttention window_size= None max_capacity_prompt= None


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ThinK attn_class= LlamaSdpaAttention window_size= None max_capacity_prompt= None


In [30]:
import pandas as pd
import glob
import os
import subprocess
from IPython.display import display, HTML

# Identify the most recent results directory
model_basename = MODEL_PATH.split('/')[-1]
result_dirs = glob.glob(f"{longbench_results_dir}/{model_basename}_*_{LAYER_STEP}_v4")

if not result_dirs:
    print("No results found to evaluate. Please run the execution pipeline above first.")
else:
    # Select the most recently created results directory
    latest_results_dir = max(result_dirs, key=os.path.getmtime)
    print(f"Evaluating results from: {latest_results_dir}")
    
    # Run the evaluation script
    # We call PyramidKV/eval.py which handles calculation across all methods and datasets
    eval_cmd = [
        "python3", "PyramidKV/eval.py",
        "--results_dir", latest_results_dir
    ]
    
    print("Calculating metrics (this may take a minute)...")
    result = subprocess.run(eval_cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        # Load and display the comparison table
        csv_path = os.path.join(latest_results_dir, "results.csv")
        if os.path.exists(csv_path):
            results_df = pd.read_csv(csv_path)
            print("\n✅ Performance Analysis Complete!")
            display(HTML("<h3>LongBench Performance Comparison</h3>"))
            display(results_df)
        else:
            print(f"Evaluation finished, but {csv_path} was not found. check output below:")
            print(result.stdout)
    else:
        print("❌ Evaluation script failed!")
        print(result.stdout)
        print(result.stderr)


Evaluating results from: ./results_long_bench/Llama-3.2-3B-Instruct_64_1_v4
Calculating metrics (this may take a minute)...
❌ Evaluation script failed!

python3: can't open file '/kaggle/working/CommonKV/PyramidKV/eval.py': [Errno 2] No such file or directory



## 4. RULER Benchmark
You can run an identical loop for RULER using `run_ruler.py` simply by swapping the command in Section 2!


In [31]:
# longbench_results_dir = "./results_ruler"
# os.makedirs(longbench_results_dir, exist_ok=True)
# MODEL_PATH = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# for method in METHODS_TO_TEST:
#     print(f"\n{'='*50}\nEvaluating Method: {method.upper()}\n{'='*50}")
    
#     # We swap 'commonkv' <-> 'ours' dynamically if user mismatched the model type
#     eval_method = method
#     if 'llama' in MODEL_PATH.lower() and method == 'ours':
#         eval_method = 'commonkv'
#     if 'mistral' in MODEL_PATH.lower() and method == 'commonkv':
#         eval_method = 'ours'

#     # # Build the command for LongBench
#     # cmd = [
#     #     "python3", "run_longbench.py",
#     #     "--method", eval_method,
#     #     "--model_path", MODEL_PATH,
#     #     "--dataset", DATASET,
#     #     "--max_capacity_prompts", str(MAX_CAPACITY_PROMPTS),
#     #     "--attn_implementation", "sdpa",  # Switch to 'eager' if FA2 fails
#     #     "--layer_step", str(LAYER_STEP),
#     #     "--rank", str(RANK),
#     #     "--save_dir", longbench_results_dir,
#     #     "--eval_batch_size", "1"
#     # ]


#     # Build the command for LongBench
#     cmd = [
#         "python3", "run_ruler.py",
#         "--method", eval_method,
#         "--model_path", MODEL_PATH,
#         "--dataset", DATASET,
#         "--max_capacity_prompts", str(MAX_CAPACITY_PROMPTS),
#         "--attn_implementation", "sdpa",
#         "--layer_step", str(LAYER_STEP),
#         "--rank", str(RANK),
#         "--save_dir", longbench_results_dir,
#         "--eval_batch_size", "1",
#         "--steps", "5",  # <--- Add this flag and target number here!
#         "--max_datasets", "2",
#         "--max_prefill_tokens_for_custom_methods", "2048"
#     ]
    
#     start_time = time.time()
    
#     # Run the subprocess
#     try:
#         # We stream output to notebook seamlessly
#         process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
#         for line in process.stdout:
#             print(line, end="")
#         process.wait()
        
#         if process.returncode == 0:
#             print(f"\n✅ Successfully completed {method} in {time.time() - start_time:.1f}s")
#         else:
#             print(f"\n❌ Failed or halted {method} (Return code: {process.returncode})")
            
#     except KeyboardInterrupt:
#         print("\nPipeline halted by user.")
#         process.kill()
#         break



In [32]:
# MODEL_PATH = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# longbench_results_dir = "./results_long_bench"
# os.makedirs(longbench_results_dir, exist_ok=True)
# METHODS_TO_TEST = ['ThinK']
# for method in METHODS_TO_TEST:
#     print(f"\n{'='*50}\nEvaluating Method: {method.upper()}\n{'='*50}")
    
#     # We swap 'commonkv' <-> 'ours' dynamically if user mismatched the model type
#     eval_method = method.lower()
#     if 'llama' in MODEL_PATH.lower() and method == 'ours':
#         eval_method = 'commonkv'
#     if 'mistral' in MODEL_PATH.lower() and method == 'commonkv':
#         eval_method = 'ours'

#     # Build the command for LongBench
#    # ... (rest of your loop configuration)

#     # Build the command for LongBench
#     cmd = [
#         "python3", "run_longbench.py",
#         "--method", eval_method,
#         "--model_path", MODEL_PATH,
#         "--dataset", DATASET,
#         # "--max_capacity_prompts", "2048",   # <--- This triggers the 4x compression for ThinK
#         "--attn_implementation", "sdpa",
#         "--save_dir", longbench_results_dir,
#         "--eval_batch_size", "1",
#         "--steps", "5",                     # Fast test on 5 samples
#         "--max_datasets", "2",
#     ]
    
    
#     start_time = time.time()
    
#     # Run the subprocess
#     try:
#         # We stream output to notebook seamlessly
#         process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
#         for line in process.stdout:
#             print(line, end="")
#         process.wait()
        
#         if process.returncode == 0:
#             print(f"\n✅ Successfully completed {method} in {time.time() - start_time:.1f}s")
#         else:
#             print(f"\n❌ Failed or halted {method} (Return code: {process.returncode})")
            
#     except KeyboardInterrupt:
#         print("\nPipeline halted by user.")
#         process.kill()
#         break



In [33]:
import torch

torch.cuda.empty_cache()


In [34]:
import torch

torch.cuda.empty_cache()
torch.cuda.ipc_collect()
